# Indexes and reading a query plan

Practice notebook for the datafromzero SQL lesson **Indexes and reading a query plan**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

The `EXPLAIN` cells print the plan with `print(...)`, because `.show()` squeezes a plan onto one line. Times in a plan are left out of the lesson: on 1,800 rows they say nothing. You add indexes only to copies, and the setup cell gives you a clean start.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it. Run it again at any time to start over.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Copy `orders` so you can add indexes

In [ ]:
con.sql("""
CREATE TABLE orders_ix AS SELECT * FROM orders
""").show()

### Step 2. Read a plan without running it

In [ ]:
print(con.execute("""
EXPLAIN SELECT * FROM orders_ix WHERE order_id = 1001
""").fetchone()[1])

### Step 3. Run it and read what happened

In [ ]:
print(con.execute("""
EXPLAIN ANALYZE SELECT * FROM orders_ix WHERE order_id = 1001
""").fetchone()[1])

### Step 4. Create the index and run it again

In [ ]:
con.sql("""
CREATE INDEX idx_orders_order_id ON orders_ix (order_id)
""").show()

In [ ]:
print(con.execute("""
EXPLAIN ANALYZE SELECT * FROM orders_ix WHERE order_id = 1001
""").fetchone()[1])

Compare with the plain `EXPLAIN` of the same query:

In [ ]:
print(con.execute("""
EXPLAIN SELECT * FROM orders_ix WHERE order_id = 1001
""").fetchone()[1])

### Step 5. A filter that cannot use the index

In [ ]:
print(con.execute("""
EXPLAIN ANALYZE SELECT * FROM orders_ix WHERE CAST(order_id AS VARCHAR) = '1001'
""").fetchone()[1])

### Step 6. A composite index

In [ ]:
con.sql("""
CREATE INDEX idx_orders_store_payment ON orders_ix (store_id, payment_method)
""").show()

In [ ]:
print(con.execute("""
EXPLAIN ANALYZE SELECT * FROM orders_ix WHERE store_id = 3 AND payment_method = 'card'
""").fetchone()[1])

### Step 7. Verify: the same answer with and without the index

In [ ]:
con.sql("""
SELECT COUNT(*) AS differing_rows FROM (
  (SELECT * FROM orders WHERE order_id = 1001 EXCEPT SELECT * FROM orders_ix WHERE order_id = 1001)
  UNION ALL
  (SELECT * FROM orders_ix WHERE order_id = 1001 EXCEPT SELECT * FROM orders WHERE order_id = 1001)
)
""").show()

## Exercise 1. Index the lookup of an order's lines

Done for you: `items_ex1`, a copy of `order_items`, and the plan before any index. Create a single-column index on `order_id`, run the same `EXPLAIN ANALYZE` again, and say what changed in the `Type` and `Row Groups Scanned` lines.

In [ ]:
con.sql("""
CREATE TABLE items_ex1 AS SELECT * FROM order_items
""").show()

In [ ]:
print(con.execute("""
EXPLAIN ANALYZE SELECT * FROM items_ex1 WHERE order_id = 1001
""").fetchone()[1])

In [ ]:
# CREATE INDEX ... ON items_ex1 (...)

In [ ]:
# The same EXPLAIN ANALYZE again, printed with print(con.execute(...).fetchone()[1])

## Exercise 2. Which filters will use the index on `customer_id`?

Done for you: `orders_ex2`, a copy of `orders` with an index on `customer_id`. Before you run any plan, write one line per filter in the next cell: `Index Scan` or `Sequential Scan`, and why. The filters are (a) `customer_id = 15`, (b) `CAST(customer_id AS VARCHAR) = '15'` and (c) `status = 'canceled'`.

In [ ]:
con.sql("""
CREATE TABLE orders_ex2 AS SELECT * FROM orders;
CREATE INDEX idx_ex2_customer ON orders_ex2 (customer_id)
""").show()

In [ ]:
# My expected result, in words:
#

In [ ]:
# (a) EXPLAIN ANALYZE ... WHERE customer_id = 15

In [ ]:
# (b) EXPLAIN ANALYZE ... WHERE CAST(customer_id AS VARCHAR) = '15'

In [ ]:
# (c) EXPLAIN ANALYZE ... WHERE status = 'canceled'

In [ ]:
# Check with EXCEPT, both ways, that (a) and (b) return the same rows

## Exercise 3. Spot the flaw

Done for you: `orders_ex3` with an index on `order_id`. A colleague runs the plain `EXPLAIN` below, sees `Sequential Scan` and drops the index. Run it, then find out what went wrong and what the colleague should have run.

In [ ]:
con.sql("""
CREATE TABLE orders_ex3 AS SELECT * FROM orders;
CREATE INDEX idx_ex3_order ON orders_ex3 (order_id)
""").show()

In [ ]:
print(con.execute("""
EXPLAIN SELECT * FROM orders_ex3 WHERE order_id = 1001
""").fetchone()[1])

In [ ]:
# What should the colleague have run? Run it here.